# 🧪 CHRUTH — Prompt Playground (Mission 3, Ollama)

Atelier pour **écrire et tester le prompt** qui dicte à l'IA comment générer
les messages de prospection (style SEKOIA).

**Comment faire :**
1. Lance la cellule *Setup*.
2. **Édite la cellule PROMPT** (rôle, consignes, garde-fous, prospect d'exemple).
3. Lance la cellule *Générer* → tu vois l'email + le script générés.
4. Répète 2–3 jusqu'à être satisfait.
5. Quand le prompt te convient : recopie `SYSTEM` + `INSTRUCTIONS` dans
   `prospect_messages.prompt_segment()` pour que tout le pipeline en profite.

**Prérequis :** Ollama lancé + modèle tiré (`ollama pull llama3.1:8b`). Données 100% locales.

## Setup

In [ ]:
import sys, pathlib, importlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import llm_client, prospect_messages as pm
importlib.reload(llm_client); importlib.reload(pm)
print('Ollama dispo :', llm_client.llm_disponible())

def _rendre(texte, p):
    for k in ('denomination', 'ville', 'effectif'):
        texte = texte.replace('{' + k + '}', str(p.get(k, '')))
    return texte

## ✏️ PROMPT — édite librement ce bloc
C'est ici que tu pilotes l'IA : le **rôle** (`SYSTEM`), les **consignes de
génération** et les **garde-fous** (`INSTRUCTIONS`), et le **prospect d'exemple**.

In [ ]:
MODELE = 'llama3.1:8b'      # modele Ollama
TEMPERATURE = 0.2          # bas = plus factuel/fiable (style SEKOIA)

# --- prospect d'exemple (pour tester le rendu des variables) ---
PROSPECT = {
    'denomination': 'CABINET DENTAIRE DU MARAIS',
    'ville': 'PARIS 03',
    'effectif': '10 a 19',
    'categorie': 'PRIV_SANTE_CABINET',
    'priorite': 'CHAUDE',
}

# --- ROLE / cadre de l'IA ---
SYSTEM = (
    "Tu es un expert en prospection commerciale B2B pour CHRUTH, societe "
    "francaise specialisee dans le nettoyage et la proprete des locaux. "
    "Tu ecris un francais professionnel, clair et concis. Tu reponds "
    'UNIQUEMENT par un objet JSON valide {"email": "...", "script": "..."}, '
    "sans aucun texte autour."
)

# --- INSTRUCTIONS : COMMENT generer (le coeur, style SEKOIA) ---
INSTRUCTIONS = '''\
Genere un email de prospection et un script d'appel telephonique pour ce segment.

SEGMENT : categorie={categorie}, priorite={priorite}

VARIABLES OBLIGATOIRES (a ecrire TEXTUELLEMENT, avec les accolades, sans les remplacer) :
- L'EMAIL doit contenir {denomination} (ex: commencer par "A l'attention de {denomination},")
  ET {ville}.
- Le SCRIPT doit contenir {denomination} ET {ville}.
- Utilise aussi {effectif} si pertinent.

CONSIGNES DE GENERATION :
- Adapte l'accroche et les arguments au SECTEUR (ex: cabinet de sante => hygiene/
  desinfection ; bureaux => espaces de travail ; commerce => surfaces clients/vitrines).
- Email : un objet court + un corps de 90 a 130 mots, signe "L'equipe CHRUTH".
- Script d'appel : 3 a 5 phrases, ton oral naturel, finissant par une demande de RDV.

GARDE-FOUS (IMPORTANT) :
- N'invente AUCUNE reference client, certification, prix ni chiffre.
- Reste credible et factuel : CHRUTH = nettoyage / proprete / entretien des locaux.
- Pas de promesses exagerees.

FORMAT : un seul objet JSON {"email": "...", "script": "..."}.
'''

## ▶️ Générer (lance après chaque édition du prompt)

In [ ]:
prompt = INSTRUCTIONS.replace('{categorie}', PROSPECT['categorie']).replace('{priorite}', PROSPECT['priorite'])
brut = llm_client.generer(prompt, SYSTEM, model=MODELE, temperature=TEMPERATURE, timeout=300)
data = pm._parser_reponse(brut)
ok = bool(data) and pm._template_valide(data)
print('source = ia OK :', ok)
print('=' * 64)
if ok:
    print('EMAIL\n'); print(_rendre(data['email'], PROSPECT))
    print('\n' + '-' * 64)
    print('SCRIPT D\'APPEL\n'); print(_rendre(data['script'], PROSPECT))
else:
    print('Sortie brute (le pipeline prendrait le repli deterministe) :\n')
    print(brut[:1000])

## ✅ Quand le prompt te convient
Recopie le contenu de `SYSTEM` et `INSTRUCTIONS` ci-dessus dans la fonction
`prompt_segment()` de `prospect_messages.py`, puis relance la génération complète
du cockpit (`GENERER_MESSAGES=True`). Tout le pipeline utilisera ton prompt validé.